# 05 — Stability Study & Final Feature Set
> **Chương 6.5** — Conservative Feature Set vs High-Performance Feature Set

Notebook này trả lời câu hỏi: **feature set nào đáng tin cậy nhất trong thực tế?**

| Feature Set | Đặc điểm | Mục tiêu |
|---|---|---|
| **Conservative** | Ít features, chỉ dùng những gì luôn có sẵn | RMSE ổn định, variance thấp |
| **High-Performance** | Đầy đủ Set 2 + XGBoost/LightGBM | RMSE thấp nhất |

**Phương pháp đo stability:** chạy 5-fold CV với 10 random seeds → boxplot phân phối RMSE.

In [ ]:
%run /content/Seminar-Pattern-Recognition/setup_colab.py

## 0. Import & Load

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import warnings; warnings.filterwarnings('ignore')

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('  xgboost không có — bỏ qua XGBRegressor')

try:
    from lightgbm import LGBMRegressor
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('  lightgbm không có — bỏ qua LGBMRegressor')

# Colab: setup_colab.py đã chạy → CWD = /content/Seminar-Pattern-Recognition
# Local: CWD có thể là notebooks/ → dùng .parent
PROJECT_ROOT = Path(os.getcwd())
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from feature_engineering import (
    rare_category_grouper, log_transform_cols,
    target_rate_encoding, add_computable_features, build_feature_matrix,
)
from evaluation import (
    evaluate_cv, print_cv_results, compare_models,
    stability_cv, compare_stability, plot_feature_importance,
)

RAW_DIR     = PROJECT_ROOT / 'data' / 'raw'
FEAT_DIR    = PROJECT_ROOT / 'data' / 'features'
MODELS_DIR  = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
SEED = 42

df_raw = pd.read_csv(RAW_DIR / 'wikicities_raw.csv')
df_raw = df_raw.dropna(subset=['population']).copy()
df_raw['log_population'] = np.log1p(df_raw['population'])
y = df_raw['log_population']
print(f' Loaded: {df_raw.shape}')

## 1. Xây dựng Conservative Feature Set

> Chỉ dùng các feature **luôn có sẵn** từ DBpedia, không cần extra SPARQL query,
> không có TRE (tránh leakage risk trong production).

In [ ]:
LOG_COLS = ['area', 'numTriples', 'abstractLen', 'populationDensity']
df_fe = log_transform_cols(df_raw, LOG_COLS)

CONSERVATIVE_COLS = [
    'log_numTriples',        # KG richness — luôn có
    'numAttributes',         # số thuộc tính — luôn có
    'hasAbstract',           # binary — luôn có
    'lat', 'lon',            # toạ độ — thường có
    'log_area',              # diện tích log
    'log_populationDensity', # density log
    'elevation',             # độ cao
    'foundingYear',          # năm thành lập
]

X_conservative = build_feature_matrix(df_fe, CONSERVATIVE_COLS)
print(f'Conservative Set: {X_conservative.shape[0]:,} × {X_conservative.shape[1]} features')
print(f'Features: {list(X_conservative.columns)}')

## 2. Rebuild High-Performance Feature Set (Set 2)

In [ ]:
feat_set2_path = FEAT_DIR / 'feature_set2.csv'
if feat_set2_path.exists():
    X_hp_full = pd.read_csv(feat_set2_path)
    y_hp = X_hp_full.pop('log_population')
    X_hp = X_hp_full
    print(f' Feature Set 2 loaded: {X_hp.shape}')
else:
    print('  feature_set2.csv chưa có — rebuild từ raw data...')
    CAT_COLS = ['country_name', 'region_name']
    for col in CAT_COLS:
        df_fe[col] = rare_category_grouper(df_fe[col], min_freq=10)
    df_tre = target_rate_encoding(
        df_train=df_fe, df_apply=df_fe,
        cat_cols=CAT_COLS, target_col='log_population', smoothing=10.0,
    )
    df_tre = add_computable_features(df_tre)
    NUM_COLS_HP = [
        'log_area', 'elevation', 'lat', 'lon', 'lat_abs',
        'log_populationDensity', 'log_area_x_density',
        'foundingYear', 'is_recent',
        'numAttributes', 'log_numTriples', 'log_triple_per_attr',
        'hasAbstract', 'log_abstractLen', 'has_region',
    ]
    TRE_COLS = ['country_name_tre', 'region_name_tre']
    X_hp = build_feature_matrix(df_tre, NUM_COLS_HP, TRE_COLS)
    y_hp = y
    print(f' High-Performance Set built: {X_hp.shape}')

## 3. Stability Analysis — 10 seeds × 5-fold CV

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)

print('Chạy stability analysis (10 seeds × 5-fold)...')
print('  Conservative Set...')
stab_conservative = stability_cv(rf, X_conservative, y, n_runs=10, cv=5, seed_start=0)

print('  High-Performance Set...')
stab_hp = stability_cv(rf, X_hp, y_hp, n_runs=10, cv=5, seed_start=0)

compare_stability(
    {
        'Conservative\n(9 features)': stab_conservative,
        'High-Performance\n(Set 2)':  stab_hp,
    },
    save_path=REPORTS_DIR / 'stability_comparison.png'
)

## 4. Multi-Model Comparison trên High-Performance Set

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest':    RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
}
if HAS_XGB:
    models['XGBoost'] = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbosity=0, n_jobs=-1,
    )
if HAS_LGB:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, verbosity=-1, n_jobs=-1,
    )

print('So sánh các model trên High-Performance Feature Set:')
comparison_df = compare_models(models, X_hp, y_hp, cv=5, seed=SEED)

print('\n Bảng tổng hợp (sắp xếp RMSE tăng dần):')
print(comparison_df[['Model', 'rmse_mean', 'rmse_std', 'r2_mean']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
bars = ax.bar(x, comparison_df['rmse_mean'],
              yerr=comparison_df['rmse_std'], capsize=5,
              color=plt.cm.tab10.colors[:len(comparison_df)], width=0.5)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=15, ha='right')
ax.set_ylabel('RMSE (log scale)')
ax.set_title('Multi-Model Comparison — High-Performance Feature Set')
for bar, val in zip(bars, comparison_df['rmse_mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'multimodel_comparison.png', bbox_inches='tight')
plt.show()

## 5. Stability của Best Model — 10 seeds

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_model      = models[best_model_name]
print(f'Best model: {best_model_name}')

stab_best = stability_cv(best_model, X_hp, y_hp, n_runs=10, cv=5, seed_start=0)

compare_stability(
    {
        'Conservative\n(RF)':           stab_conservative,
        'HP Set\n(RF)':                 stab_hp,
        f'HP Set\n({best_model_name})': stab_best,
    },
    save_path=REPORTS_DIR / 'stability_full_comparison.png'
)

## 6. Feature Importance — Best Model

In [ ]:
best_model.fit(X_hp, y_hp)

if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=X_hp.columns)
    importance = importance.sort_values(ascending=False)
    plot_feature_importance(
        importance, top_n=20,
        title=f'Feature Importance — {best_model_name} (HP Set)',
        save_path=REPORTS_DIR / 'feature_importance_best.png'
    )
    print('Top 10 features:')
    print(importance.head(10).to_string())

## 7. Tổng kết & Khuyến nghị

In [ ]:
cons_rmse = stab_conservative.groupby('run')['rmse'].mean()
hp_rmse   = stab_hp.groupby('run')['rmse'].mean()

print('=' * 60)
print('  TỔNG KẾT — STABILITY STUDY')
print('=' * 60)
print(f"""
Conservative Feature Set ({X_conservative.shape[1]} features):
  RMSE mean : {cons_rmse.mean():.4f}
  RMSE std  : {cons_rmse.std():.4f}   ← stability
  Ưu điểm  : Không cần extra query, không data leakage risk
  Nhược điểm: RMSE cao hơn

High-Performance Feature Set ({X_hp.shape[1]} features):
  RMSE mean : {hp_rmse.mean():.4f}
  RMSE std  : {hp_rmse.std():.4f}   ← stability
  Ưu điểm  : RMSE thấp hơn, TRE giữ thông tin quốc gia
  Nhược điểm: TRE cần careful CV để tránh leakage

Khuyến nghị:
  - Production / deployment  → Conservative Set (robust, predictable)
  - Competition / research   → HP Set + {best_model_name}
""")
print('=' * 60)

## 8. Lưu Final Model & Artifacts

In [ ]:
# Lưu best model
joblib.dump(best_model, MODELS_DIR / 'best_model.pkl')
print(f' best_model.pkl  → {MODELS_DIR}')

# Metadata
metadata = {
    'model_name':    best_model_name,
    'feature_set':   'high_performance',
    'n_features':    int(X_hp.shape[1]),
    'feature_names': list(X_hp.columns),
    'cv_rmse_mean':  float(hp_rmse.mean()),
    'cv_rmse_std':   float(hp_rmse.std()),
    'n_train':       int(len(y_hp)),
}
with open(MODELS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f' model_metadata.json lưu xong')

# Conservative model
rf_cons = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_cons.fit(X_conservative, y)
joblib.dump(rf_cons, MODELS_DIR / 'conservative_model.pkl')
print(f' conservative_model.pkl lưu xong')

print('\n Hoàn tất toàn bộ pipeline Feature Engineering!')